In [ ]:
#Load from the datasets the elements needed 
import pandas as pd
import re

#load the .ods file
df = pd.read_excel("C:\\Users\\Bryan\\Downloads\\88milSMS_88522.ods", engine="odf")  #uses 'odf' engine to read .ods files

#check available columns
print("Available columns:", df.columns)

# Make sure the column containing the SMS is correctly identified
sms_column = "SMS_ANON"  # Replace with the exact name of the column containing the SMS
if sms_column not in df.columns:
    raise ValueError(f"La colonne '{sms_column}' n'existe pas dans le fichier.")

# Remove duplicates and null values
df = df.drop_duplicates(subset=[sms_column])
df = df[df[sms_column].notnull()]
df = df[df[sms_column].str.strip().str.len() > 0]

# --- INFORMALITY FILTER ---
# Informal keyword list
informal_keywords = [
    "mdr", "lol", "ptdr", "wesh", "stp", "svp", "pk", "tfk", "svp", "tkt", "bb",
    "goumin", "hein", "ehh", "frr", "ss", "ko", "gâté", "walaï", "walai",
    "gars", "bg", "t’es", "t ke", "jvai", "c koi", "c kwa", "neeeh", "ehh", "chai",
    "ouuu", "yen a", "ahh", "dcp", "vrm", "nan", "koi", "dja", "gna", "man",
    "gnin", "franchement", "ohhhh", "ehh", "laaarge", "suis", "nan", "bah", "vs",
    "tt", "ouai", "wsh", "dsl", "ds", "mec", "flemme", "chier", "lsp", "bsx", "nn",
    "ki", "tkt", "rt", "azy", "meuf", "ke", "ben", "pa", "sinn", "couilles", "bcp",
    "b1", "tjs", "qql", "qques", "qlq", "tjrs"
]

# Emoji detection pattern
emoji_pattern = r"[😀-🙏🙌❤️😍😒😂🤣😅😊😎😉😘🤦🤷👌👍👀]"

# Function to detect informal sentences
def is_informal(text):
    text_lower = text.lower()
    
    # Emoji detection
    if re.search(emoji_pattern, text):
        return True
    
    # Informal keyword detection
    for kw in informal_keywords:
        if kw in text_lower:
            return True
    
    # Short text (SMS style)
    if len(text.split()) <= 5:
        return True
    
    # Presence of repeated characters (informal style)
    if re.search(r"(.)\1{2,}", text_lower):
        return True
    
    return False

# Apply the informality filter
df["is_informal"] = df[sms_column].apply(is_informal)

# Keep only informal sentences
informal_df = df[df["is_informal"] == True]

# If the number of informal sentences is insufficient, include semi-informal sentences
if len(informal_df) < 50000:
    semi_informal = df[(df["is_informal"] == False) & (df[sms_column].str.split().str.len() < 12)]
    informal_df = pd.concat([informal_df, semi_informal]).drop_duplicates().reset_index(drop=True)

# Extract a random sample of 50,000 sentences
final_sample = informal_df.sample(n=50000, random_state=42)

# --- FILE SAVING ---
# Save the informal sentences to a CSV file
final_sample[sms_column].to_csv("daft_sms_50k.csv", index=False, encoding="utf-8")
print("Dataset DAFT SMS 50k generated successfully.")
print("Final size =", len(final_sample))

In [ ]:
import pandas as pd
import re
from unidecode import unidecode

# Load the dataset
df = pd.read_csv("daft_sms_50k.csv", encoding="utf-8", sep=";", on_bad_lines='skip')

# Check available columns
print("Available columns:", df.columns)
# Make sure the column containing the SMS messages is correctly identified
sms_column = "text"  # Replace with the exact name of the column containing the SMS messages
if sms_column not in df.columns:
    raise ValueError(f"The column '{sms_column}' does not exist in the file.")

# --- 1. CORRECT SPECIAL CHARACTERS ---
# Function to correct special characters
def clean_special_characters(text):
    if not isinstance(text, str):  # Check if the text is a string
        return text  # Return the value as is if it's not a string
    # Replace misencoded characters with their equivalents
    text = text.replace("Ã©", "é").replace("Ã¨", "è").replace("Ã", "à").replace("Ã´", "ô").replace("Ãê", "ê")
    text = text.replace("Ã§", "ç").replace("Ã»", "û").replace("Ã¹", "ù").replace("Ãî", "î").replace("Ã¤", "ä")
    text = text.replace("Ã¶", "ö").replace("Ã¼", "ü").replace("Ã€", "À").replace("Ã‰", "É").replace("Ãˆ", "È")
    # Remove remaining accents with unidecode
    text = unidecode(text)
    return text

df[sms_column] = df[sms_column].apply(clean_special_characters)
# --- 2. REMOVE TEXT-BASED EMOJIS ---
# List of text-based emojis
text_emojis = [
    "=)", ";-)", ":=)", ":D", "xD", ":P", ":)", ":(", ":-D", ":-)", ":-(", "<3", ":*", "=)", "=("
]

# Function to remove text-based emojis
def remove_text_emojis(text):
    if not isinstance(text, str):  # Check if the text is a string
        return text  # Return the value as is if it's not a string
    for emoji in text_emojis:
        text = text.replace(emoji, "")
    return text.strip()

df[sms_column] = df[sms_column].apply(remove_text_emojis)

# --- 3. REMOVE TOO FORMAL SENTENCES ---
# List of formal keywords
formal_keywords = [
    "cordialement", "néanmoins", "monsieur", "madame", "s'il vous plaît", "pourriez-vous",
    "cependant", "Veuillez cliquer sur le lien suivant", 
    "chère", "bien à vous", "je vous prie", "toutefois", "conformément",
    "facture", "Ceci est un message automatique", "objet", "référence", 
    "je reste à votre disposition", "Votre paiement a été accepté", 
    "Nous vous informons que","Votre commande est prête","client",
    "procédure", "Merci d’avoir répondu à notre message", "Votre code de confirmation est", 
    "service client", "avis", "je vous remercie", "merci d’avoir"
    "merci d’avoir", "veuillez","votre compte" ,"votre paiement" ,"votre abonnement" ,"votre facture" ,"votre commande" ,"votre code",
    "nous vous informons", "assistance" ,"ce message ne nécessite pas de réponse" ,"ne pas répondre" ,"clickez/clicker sur le lien",
    "message automatique"
]

# Function to detect formal sentences
def is_formal(text):
    if not isinstance(text, str):  # Check if the text is a string
        return False  # Consider non-strings as not formal
    text_lower = text.lower()
    for kw in formal_keywords:
        if kw in text_lower:
            return True
    return False


def anonymize_names(text):
    if not isinstance(text, str):
        return text
    
    text = re.sub(r'\*[^*]+\*', 'Anon', text, flags=re.IGNORECASE)
    
    return text.strip()

# Apply anonymization
df[sms_column] = df[sms_column].apply(anonymize_names)

# Remove too formal sentences
df = df[~df[sms_column].apply(is_formal)]

df = df.dropna(axis=1, how="all")  # Remove entirely empty columns

# --- 4. SAVE THE CLEANED DATASET ---
# Save the cleaned dataset to a CSV file
df.to_csv("cleaned_daft_sms_50k.csv", index=False, encoding="utf-8")
print("Cleaned dataset saved successfully.")
print("Final dataset size:", len(df))
print(df.head())

In [23]:
import pandas as pd
import random
import re
from nltk.corpus import wordnet as wn
import nltk
nltk.download('omw-1.4')
nltk.download('wordnet')

# -------------------------------------------------------
# LOAD YOUR DATASET
# -------------------------------------------------------

try:
    df = pd.read_csv("cleaned_daft_sms_50k.csv", sep=";", on_bad_lines='skip', encoding="utf-8")
    print("File loaded successfully.")
except Exception as e:
    print(f"Error loading file : {e}")

texts = df["text"].astype(str).tolist()


# -------------------------------------------------------
# 1. SYNONYM REPLACEMENT (FREE + OFFLINE)
# -------------------------------------------------------
def get_french_synonyms(word):
    syns = []
    for syn in wn.synsets(word, lang='fra'):
        for lemma in syn.lemma_names('fra'):
            if lemma.lower() != word.lower():
                syns.append(lemma.replace("_", " "))
    return list(set(syns))

def synonym_replacement(sentence, n=1):
    words = sentence.split()
    new_words = words.copy()
    
    random_words = list(set(words))
    random.shuffle(random_words)
    
    count = 0
    for word in random_words:
        synonyms = get_french_synonyms(word)
        if synonyms:
            synonym = random.choice(synonyms)
            new_words = [synonym if w == word else w for w in new_words]
            count += 1
        if count >= n:
            break
    return " ".join(new_words)


# -------------------------------------------------------
# 2. RANDOM SWAP
# -------------------------------------------------------
def random_swap(sentence):
    words = sentence.split()
    if len(words) < 2:
        return sentence
    idx1, idx2 = random.sample(range(len(words)), 2)
    words[idx1], words[idx2] = words[idx2], words[idx1]
    return " ";join(words)


# -------------------------------------------------------
# 3. RANDOM DELETION
# -------------------------------------------------------
def random_deletion(sentence, p=0.1):
    words = sentence.split()
    if len(words) == 1:
        return sentence
    new_words = [w for w in words if random.random() > p]
    if len(new_words) == 0:
        return random.choice(words)
    return " ".join(new_words)


# -------------------------------------------------------
# 4. NOISE INJECTION (SMS-STYLE)
# -------------------------------------------------------
def noise_injection(sentence):
    sentence = sentence.replace("ou", "oo")
    sentence = sentence.replace("ah", "ahh")
    if random.random() < 0.5:
        sentence += random.choice([" !!", " !!!", " !!!!!!!", " !???"])
    return sentence


# -------------------------------------------------------
# MAIN AUGMENTATION LOOP
# -------------------------------------------------------
augmented = []

for text in texts[:5000]:  # Apply augmentation to 5,000 messages
    try:
        aug1 = synonym_replacement(text)
        aug2 = random_swap(text)
        aug3 = random_deletion(text)
        aug4 = noise_injection(text)
        
        augmented += [text, aug1, aug2, aug3, aug4]
    except:
        augmented.append(text)

# Make dataframe
aug_df = pd.DataFrame({"SMS": list(set(augmented))})

# Save augmented dataset
aug_df.to_csv("augmented_daft_sms.csv", index=False)

print("Augmented dataset created:", len(aug_df))


[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Bryan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Bryan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Fichier chargé avec succès.
Augmented dataset created: 18291
Augmented dataset created: 18291


In [53]:
import pandas as pd
import random


def elongate_words(sentence, elongation_prob=0.5):  
    words = sentence.split()
    for i in range(len(words)):
        word = words[i]
        elongated_word = ""
        for char in word:
            elongated_word += char
            if char in "aeiouyAEIOUY": 
                elongated_word += char * random.randint(1, 3)
        words[i] = elongated_word
    return " ".join(words)


try:
    df = pd.read_csv("corpus_maitre.csv", sep=";", on_bad_lines='skip', encoding="utf-8")
    print("File loaded successfully.")
except Exception as e:
    print(f"Error loading file: {e}")
# Verify the column containing the SMS
sms_column = "text"  # Replace with the exact name of the column containing the SMS
if sms_column not in df.columns:
    raise ValueError(f"Column  '{sms_column}'doesnt exist.")

# Apply elongation
df["elongated_text"] = df[sms_column].apply(elongate_words)

# Combine original and elongated sentences to augment data
augmented_df = pd.concat([df[[sms_column]], df[["elongated_text"]].rename(columns={"elongated_text": sms_column})])

# Save the augmented dataset
augmented_df.to_csv("corpus_main_vd.csv", index=False, encoding="utf-8")
print("Dataset augmented and saved.")

File loaded successfully.
Dataset augmented and saved.


In [57]:
import nlpaug.augmenter.char as nac
import pandas as pd

# Verify the column containing the SMS
sms_column = "text"  # Replace with the exact name of the column containing the SMS
if sms_column not in df.columns:
    raise ValueError(f"Column '{sms_column}' doesn't exist. Available columns: {df.columns}")

# Initialize the KeyboardAug augmenter
try:
    aug = nac.KeyboardAug(aug_char_p=0.3)  # Probability of character typo
    print("KeyboardAug initialized successfully.")
except Exception as e:
    raise Exception(f"An error occurred while initializing KeyboardAug: {e}")

# Apply the augmenter to the dataset
try:
    augmented_texts = df[sms_column].apply(aug.augment)
    print("Augmentation applied successfully.")
except Exception as e:
    raise Exception(f"An error occurred during augmentation: {e}")

# Save only the augmented responses to the dataset
try:
    augmented_df = pd.DataFrame({sms_column: augmented_texts})
    augmented_df.to_csv("corpus_main_augmented_spelling_error.csv", index=False, encoding="utf-8")
    print("Augmented dataset saved successfully.")
except Exception as e:
    raise Exception(f"An error occurred while saving the dataset: {e}")

KeyboardAug initialized successfully.
Augmentation applied successfully.
Augmented dataset saved successfully.


In [60]:
import nlpaug.augmenter.word as naw  # Import the 'naw' module for word-level augmenters
import pandas as pd

# Load the dataset
df = pd.read_csv("corpus_maitre.csv", sep=";", on_bad_lines='skip', encoding="utf-8")

# Verify the column containing the SMS
sms_column = "text"  # Replace with the exact name of the column containing the SMS
if sms_column not in df.columns:
    raise ValueError(f"Column '{sms_column}' doesn't exist. Available columns: {df.columns}")

# Parameters for augmentation
TOPK = 20  # default=100
ACT = 'insert'  # "substitute"

# Initialize the ContextualWordEmbsAug augmenter
try:
    aug_bert = naw.ContextualWordEmbsAug(
        model_path='distilbert-base-uncased', 
        # device='cuda',  # Uncomment this if you want to use GPU
        action=ACT, 
        top_k=TOPK
    )
    print("ContextualWordEmbsAug initialized successfully.")
except Exception as e:
    raise Exception(f"An error occurred while initializing ContextualWordEmbsAug: {e}")

# Apply the augmenter to the dataset
try:
    augmented_data = []
    for text in df[sms_column]:
        augmented_variants = [aug_bert.augment(text) for _ in range(5)]  # Generate 5 variants per sentence
        augmented_data.extend(augmented_variants)
    print("Augmentation applied successfully.")
except Exception as e:
    raise Exception(f"An error occurred during augmentation: {e}")

# Save only the augmented responses to the dataset
try:
    augmented_df = pd.DataFrame({sms_column: augmented_data})
    augmented_df.to_csv("corpus_main_augmented_contextual.csv", index=False, encoding="utf-8")
    print("Augmented dataset saved successfully.")
except Exception as e:
    raise Exception(f"An error occurred while saving the dataset: {e}")

ContextualWordEmbsAug initialized successfully.
Augmentation applied successfully.
Augmented dataset saved successfully.
Augmentation applied successfully.
Augmented dataset saved successfully.


In [1]:
import pandas as pd
import random

# Function for random word dropout
def random_word_dropout(text):
    words = text.split()
    if len(words) > 1:
        words.pop(random.randint(0, len(words) - 1))  # Remove a random word
    return " ".join(words)

# Function to generate 9 variations per line using random word dropout
def generate_variants(corpus):
    variants = []
    
    # Loop through each line in the corpus
    for line in corpus:
        line_variants = []
        
        # Generate 9 variations for each line
        for _ in range(9):
            line_variants.append(random_word_dropout(line))  # Apply dropout to each line
            
        variants.append(line_variants)
    
    return variants

# Load your dataset (make sure to adjust the path to your file)
df = pd.read_csv("corpus_maitre.csv", sep=";", on_bad_lines='skip', encoding="utf-8")

# Assuming the relevant text column is named 'text', if not, adjust it to the correct column name
corpus = df['text'].tolist()

# Generate variants for the entire dataset
generated_variants = generate_variants(corpus)

# Flatten the list of generated variants into a single list for saving in a CSV
flat_variants = [item for sublist in generated_variants for item in sublist]

# Save the generated variants into a new CSV file, keeping only the output (variations)
output_df = pd.DataFrame(flat_variants, columns=["generated_text"])

# Save the output CSV
output_df.to_csv('dropout_augmented_data.csv', index=False)

print("The augmented data has been saved as 'dropout_augmented_data.csv'.")

The augmented data has been saved as 'dropout_augmented_data.csv'.


In [2]:
import pandas as pd
import random

# Function for random word insertion
def random_word_insertion(text):
    words = text.split()
    
    # List of words to randomly insert into the text (you can customize this list)
    insert_words = ['lol', 'mdr', 'tkt', 'frr', 'chill', 'wsh', 'vibes', 'nan', 'tkt', 
                    'nptq' , 'yo','Ouf', 'Relou', 'Chelou', 'Chanmé', 'Thune', 'fric', 
                    'oseille', 'blé', 'Balles', 'Mec', 'gars', 'type', 'Meuf', 'nana', 
                    'Pote', 'Frangin', 'frangine', 'Beauf', 'Kiffer', 'Taffer', 'bosser', 
                    'Bouffer', 'grailler', 'Glander', 'Piger', 'Bouffe', 'Baraque', 'Caisse', 
                    'bagnole', 'Boulot', 'taf', 'Pcq', 'Psk', 'Pk', 'Pkoi', 'Jpp', 'Ptdr', 'Xptdr', 
                    'Oklm', 'Osef', 'Blc', 'Mrc', 'Dsl', 'Stp', 'Svp', 'Dac', 'Ok', 'Jtm', 'Re', 'Afk',
                    'Bjr', 'Bsr', 'CC', 'A+', 'IDK', 'OMG']
    
    # Randomly select an index to insert a word
    if len(words) > 1:  # Ensure the sentence has more than one word
        insert_index = random.randint(0, len(words) - 1)
        word_to_insert = random.choice(insert_words)
        words.insert(insert_index, word_to_insert)  # Insert the random word
    
    return " ".join(words)

# Function to generate 9 variations per line using random word insertion
def generate_variants(corpus):
    variants = []
    
    # Loop through each line in the corpus
    for line in corpus:
        line_variants = []
        
        # Generate 9 variations for each line
        for _ in range(9):
            line_variants.append(random_word_insertion(line))  # Apply random insertion to each line
            
        variants.append(line_variants)
    
    return variants

# Load your dataset (make sure to adjust the path to your file)
df = pd.read_csv("corpus_maitre.csv", sep=";", on_bad_lines='skip', encoding="utf-8")

# Assuming the relevant text column is named 'text', if not, adjust it to the correct column name
corpus = df['text'].tolist()

# Generate variants for the entire dataset
generated_variants = generate_variants(corpus)

# Flatten the list of generated variants into a single list for saving in a CSV
flat_variants = [item for sublist in generated_variants for item in sublist]

# Save the generated variants into a new CSV file, keeping only the output (variations)
output_df = pd.DataFrame(flat_variants, columns=["generated_text"])

# Save the output CSV
output_df.to_csv('augmented_data_with_insertion.csv', index=False)

print("The augmented data with random word insertion has been saved as 'augmented_data_with_insertion.csv'.")


The augmented data with random word insertion has been saved as 'augmented_data_with_insertion.csv'.


In [3]:
import pandas as pd
import random

# Function for sentence shuffling (shuffle the words in the sentence)
def sentence_shuffling(text):
    words = text.split()
    
    # Shuffle the words in the sentence
    random.shuffle(words)
    
    return " ".join(words)

# Function to generate 9 variations per line using sentence shuffling
def generate_variants(corpus):
    variants = []
    
    # Loop through each line in the corpus
    for line in corpus:
        line_variants = []
        
        # Generate 9 variations for each line
        for _ in range(9):
            line_variants.append(sentence_shuffling(line))  # Apply sentence shuffling to each line
            
        variants.append(line_variants)
    
    return variants

# Load your dataset (make sure to adjust the path to your file)
df = pd.read_csv("corpus_maitre.csv", sep=";", on_bad_lines='skip', encoding="utf-8")

# Assuming the relevant text column is named 'text', if not, adjust it to the correct column name
corpus = df['text'].tolist()

# Generate variants for the entire dataset
generated_variants = generate_variants(corpus)

# Flatten the list of generated variants into a single list for saving in a CSV
flat_variants = [item for sublist in generated_variants for item in sublist]

# Save the generated variants into a new CSV file, keeping only the output (variations)
output_df = pd.DataFrame(flat_variants, columns=["generated_text"])

# Save the output CSV
output_df.to_csv('augmented_data_with_shuffling.csv', index=False)

print("The augmented data with sentence shuffling has been saved as 'augmented_data_with_shuffling.csv'.")


The augmented data with sentence shuffling has been saved as 'augmented_data_with_shuffling.csv'.


In [5]:
import pandas as pd
import random
import nlpaug.augmenter.word as naw

# -------------------------------------------------------
# LOAD YOUR DATASET
# -------------------------------------------------------
try:
    df = pd.read_csv("corpus_maitre.csv", sep=";", on_bad_lines='skip', encoding="utf-8")
    print("File loaded successfully.")
except Exception as e:
    print(f"Error loading file: {e}")

texts = df["text"].astype(str).tolist()


# -------------------------------------------------------
# SPLIT AUGMENTATION - USING naw.SplitAug
# -------------------------------------------------------

# Initialize the SplitAugmentation with your desired parameters
aug = naw.SplitAug(name='Split_Aug', aug_min=1, aug_max=10, aug_p=0.3, min_char=4, stopwords=None, tokenizer=None, 
                   reverse_tokenizer=None, stopwords_regex=None, verbose=0)


# -------------------------------------------------------
# MAIN AUGMENTATION LOOP (ONLY CONTRADICTIONS)
# -------------------------------------------------------

augmented = []

# Generate 9 variations for each sentence (limiting to 5000 texts as a sample)
for text in texts[:5000]:
    try:
        # Generate 9 augmentations for each text
        for _ in range(9):
            # Apply SplitAugmentation to each sentence
            aug_text = aug.augment(text)
            augmented.append(aug_text[0])  # Extract the first generated variant (if there are multiple)

    except Exception as e:
        print("Error on text:", text)
        print("Exception:", e)

# Make final dataframe with the augmented data (only outputs)
aug_df = pd.DataFrame({"text": list(set(augmented))})

# Save new augmented dataset (only the outputs)
aug_df.to_csv("augmented_corpus_with_split_augmentation.csv", index=False)

print("Augmented dataset created:", len(aug_df))


File loaded successfully.
Augmented dataset created: 6728
Augmented dataset created: 6728


In [ ]:
import tweepy
import pandas as pd
import re

# Replace these variables with your own API keys
BEARER_TOKEN = "AAAAAAAAAAAAAAAAAAAAALNV5QEAAAAAUrM9zeBtusP7uUzIttlpq1wLWHE%3DE66MT3rj918YeTRV4wW7CeX97VPMONXyK1dOT6Oftu2jDkxrfc"

# Authenticate with the Twitter API v2
client = tweepy.Client(bearer_token=BEARER_TOKEN)

# Function to clean tweets (remove URLs, mentions, etc.)
def clean_tweet(tweet):
    tweet = tweet.lower()
    tweet = tweet.replace("\n", " ").replace("\r", " ")
    tweet = re.sub(r"http\S+", "", tweet)  # Remove URLs
    tweet = re.sub(r"@\S+", "", tweet)  # Remove mentions
    tweet = re.sub(r"#\S+", "", tweet)  # Remove hashtags
    tweet = re.sub(r"[^a-zA-Zéèàçùâêîôû\s]", "", tweet)  # Remove special characters
    return tweet.strip()

# Collect tweets with specific hashtags or keywords
query = "(#mdr OR wsh OR azy OR flemme OR belec or nan or tkt or oklm or frero or slangFR) lang:fr -is:retweet"  # Search query
tweets = client.search_recent_tweets(query=query, max_results=10, tweet_fields=["text"])

# List to store cleaned tweets
tweets_data = []
for tweet in tweets.data:
    clean_text = clean_tweet(tweet.text)
    if len(clean_text.split()) > 3:  # Filter out very short tweets
        tweets_data.append(clean_text)

# Convert the tweets to a DataFrame and save them
tweets_df = pd.DataFrame(tweets_data, columns=["SMS"])
tweets_df.to_csv("tweets_informels_1000.csv", index=False)
print(f"{len(tweets_data)} tweets collected and saved.")

In [10]:
import pandas as pd

# Liste des fichiers que tu veux concaténer
files = [
    "corpus_main_vd.csv", 
    "augmented_corpus.csv", 
    "corpus_augmented_spelling_error.csv",   
    "corpus_main_augmented_contextual_2.csv",
    "corpus_main_augmented_contextual_Bert.csv",
    "sampling_corpus.csv",
    "tweets_informels.csv",
    "augmented_corpus_with_split_augmentation.csv",
    "augmented_data_with_insertion.csv",
    "augmented_data_with_shuffling.csv",
    "dropout_augmented_data.csv"
]


dfs = []


for file in files:
    df = pd.read_csv(file, sep=";", on_bad_lines='skip', encoding="utf-8")  # Load the file
    if "text" in df.columns:  # Check if the "text" column exists
        dfs.append(df[["text"]])  # Take only the "text" column

# Concatenate all DataFrames into a single DataFrame
final_df = pd.concat(dfs, ignore_index=True)

final_df = final_df.drop_duplicates(subset=["text"])  #remove duplicates
final_df = final_df.dropna(subset=["text"])  #remove empty lines


final_df.to_csv("final_corpus.csv", index=False)

print(f"Final dataset ready with {len(final_df)} elements.")


Final dataset ready with 100585 elements.


In [12]:
import pandas as pd

# Liste des fichiers à concaténer
files = [
    "sts_dataset_random_pairing.csv",
    "sts_dataset_backtranslation_vd.csv",
    "sts_dataset_hard_negative.csv"
]

dfs = []

for file in files:
    df = pd.read_csv(file, sep=";", on_bad_lines='skip', encoding="utf-8")
    # Vérifie que les colonnes nécessaires existent
    required_cols = {"sentence1", "sentence2", "similarity_score"}
    if required_cols.issubset(df.columns):
        # On force la colonne similarity_score à être vide
        df["similarity_score"] = ""
        dfs.append(df[["sentence1", "sentence2", "similarity_score"]])
    else:
        print(f"Attention: Colonnes manquantes dans {file}, ignoré.")

# Concaténation
final_df = pd.concat(dfs, ignore_index=True)

# Suppression des doublons et des lignes vides
final_df = final_df.drop_duplicates(subset=["sentence1", "sentence2"])
final_df = final_df.dropna(subset=["sentence1", "sentence2"])

# Sauvegarde du corpus final
final_df.to_csv("sts_augmented_without_score.csv", sep=";", index=False, encoding="utf-8")

print(f"Final dataset ready with {len(final_df)} pairs.")

Final dataset ready with 9000 pairs.


In [15]:
import pandas as pd

# Liste des fichiers à concaténer
files = [
    "sts_dataset_annotated.csv",
    "STS_dataset.csv"
 ]

dfs = []

for file in files:
    df = pd.read_csv(file, sep=";", on_bad_lines='skip', encoding="utf-8")
    # Vérifie que les colonnes nécessaires existent
    required_cols = {"sentence1", "sentence2", "similarity_score"}
    if required_cols.issubset(df.columns):
        dfs.append(df[["sentence1", "sentence2", "similarity_score"]])
    else:
        print(f"Warning : Column missing in {file}, ignored.")

# Concaténation
final_df = pd.concat(dfs, ignore_index=True)

# Suppression des doublons et des lignes vides
final_df = final_df.drop_duplicates(subset=["sentence1", "sentence2"])
final_df = final_df.dropna(subset=["sentence1", "sentence2"])

 # Sauvegarde du corpus final
final_df.to_csv("sts_finetuning_dataset.csv", sep=";", index=False, encoding="utf-8")

print(f"Final dataset ready with {len(final_df)} pairs.")

Final dataset ready with 9427 pairs.


In [2]:
#Random pairing for STS dataset creation
import pandas as pd
import random

# -----------------------------
# SETTINGS
# -----------------------------
INPUT_FILE = "final_corpus.csv"      # ton dataset de 100k phrases
OUTPUT_FILE = "sts_dataset_random_pairing.csv"
NUM_PAIRS = 5000                     # nombre de paires à générer

# -----------------------------
# LOAD DATASET
# -----------------------------
print("Loading dataset...")
df = pd.read_csv(INPUT_FILE, sep=";", encoding="utf-8", on_bad_lines="skip")

# On s'assure que la colonne 'text' existe (à adapter si ton corpus a un autre nom)
if "text" not in df.columns:
    raise ValueError("The input dataset must contain a column named 'text'.")

sentences = df["text"].astype(str).tolist()

if len(sentences) < 2:
    raise ValueError("Dataset must contain at least 2 sentences.")

# -----------------------------
# GENERATE RANDOM PAIRS
# -----------------------------
print(f"Generating {NUM_PAIRS} random pairs...")

pairs_sentence1 = []
pairs_sentence2 = []

for _ in range(NUM_PAIRS):
    s1, s2 = random.sample(sentences, 2)  # choisit 2 phrases distinctes
    pairs_sentence1.append(s1)
    pairs_sentence2.append(s2)

# -----------------------------
# BUILD OUTPUT DATAFRAME
# -----------------------------
output_df = pd.DataFrame({
    "sentence1": pairs_sentence1,
    "sentence2": pairs_sentence2,
    "similarity_score": [""] * NUM_PAIRS   # colonne vide
})

# -----------------------------
# SAVE CSV UTF-8 avec séparateur ;
# -----------------------------
output_df.to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print(f"Done! File saved as {OUTPUT_FILE}")
print(output_df.head())

Loading dataset...
Generating 5000 random pairs...
Done! File saved as sts_dataset_random_pairing.csv
                                           sentence1  \
0       Bon...je dois aller manger...je te laisse,,,   
1  Aucune idee je suis dans le 6173, je rentre de...   
2                                       Dans 2min,,,   
3                          C'est ca oui ! Matcho.,,,   
4                     A LLEZ T OUS vs FAIRE ENCULER!   

                                           sentence2 similarity_score  
0  Pas contre se soir je risque d'aatre a Montpel...                   
1  je souhaite etre la pr toi Jpp qd tu en auras ...                   
2   Peut etre, bon faut vraiment que jvy aille...,,,                   
3        qui Y'a ton ma frÃ¨re t'appelle chatte. sur                   
4  On t attend ou on mange, parce que la viande v...                   


In [ ]:
#Verification of the STS scores repartition on the dataset
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -----------------------------
# SETTINGS
# -----------------------------
MODEL_PATH = "./flaubert_sts_final"   # chemin vers ton modèle STS fine-tuné
STS_DATASET_FILE = "STS_dataset.csv"  # dataset sur lequel on vérifie
MAX_SAMPLES = None                    # mettre 5000 si tu veux échantillonner

# -----------------------------
# LOAD MODEL
# -----------------------------
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# -----------------------------
# LOAD DATASET
# -----------------------------
print("Loading STS dataset...")
df = pd.read_csv(STS_DATASET_FILE, sep=";", encoding="utf-8", on_bad_lines="skip")

df = df.dropna(subset=["sentence1", "sentence2"])

if MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES)

# -----------------------------
# FUNCTION FOR PREDICTING SCORES
# -----------------------------
def predict_score(sentence1, sentence2):
    inputs = tokenizer(
        sentence1, sentence2,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        score = outputs.logits.squeeze().item()

    # limit final values to [0, 5]
    return max(0, min(5, score))

# -----------------------------
# PREDICT ALL SCORES
# -----------------------------
print("Predicting STS scores...")
predicted_scores = df.apply(
    lambda row: predict_score(row["sentence1"], row["sentence2"]), axis=1
)

df["predicted_score"] = predicted_scores

# -----------------------------
# STATS & DISTRIBUTION PLOT
# -----------------------------
print("\n===== STATS =====")
print(df["predicted_score"].describe())

score_counts = df["predicted_score"].round().value_counts().sort_index()

print("\nScore distribution (rounded to nearest integer):")
print(score_counts)

# -----------------------------
# HISTOGRAM + DENSITY PLOT
# -----------------------------
plt.figure(figsize=(10, 6))
plt.hist(df["predicted_score"], bins=30, alpha=0.7, color="skyblue")
plt.title("Distribution des scores STS prédits")
plt.xlabel("Score STS (0 → 5)")
plt.ylabel("Nombre de paires")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

# -----------------------------
# SAVE RESULTS
# -----------------------------
df.to_csv("STS_predicted_with_scores.csv", sep=";", index=False, encoding="utf-8")
print("\nSaved predictions to STS_predicted_with_scores.csv")

In [7]:
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
import random
from tqdm import tqdm

# -----------------------------
# SETTINGS
# -----------------------------
INPUT_FILE = "final_corpus.csv"
OUTPUT_FILE = "sts_dataset_backtranslation_vd.csv"
NUM_PAIRS = 2000                # number of pairs to generate
BATCH_SIZE = 8                  # batch size for speed

# -----------------------------
# LOAD DEVICE (USE GPU IF AVAILABLE)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# LOAD DATASET
# -----------------------------
print("\nLoading dataset...")
df = pd.read_csv(INPUT_FILE, sep=";", encoding="utf-8", on_bad_lines="skip")

if "text" not in df.columns:
    raise ValueError("The dataset must contain a column named 'text'.")

sentences = df["text"].astype(str).tolist()

# Random selection of sentences
original_sentences = random.sample(sentences, NUM_PAIRS)

# -----------------------------
# LOAD TRANSLATION MODELS
# -----------------------------
print("\nLoading translation models (FR→EN and EN→FR)...")

# FR → EN
model_fr_en_name = "Helsinki-NLP/opus-mt-fr-en"
tokenizer_fr_en = MarianTokenizer.from_pretrained(model_fr_en_name)
model_fr_en = MarianMTModel.from_pretrained(model_fr_en_name).to(device)

# EN → FR
model_en_fr_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer_en_fr = MarianTokenizer.from_pretrained(model_en_fr_name)
model_en_fr = MarianMTModel.from_pretrained(model_en_fr_name).to(device)


# -----------------------------
# BATCH TRANSLATION FUNCTION
# -----------------------------
def translate_batch(model, tokenizer, sentences):
    inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]

def backtranslate_batch(batch_sentences):
    # FR -> EN
    en_batch = translate_batch(model_fr_en, tokenizer_fr_en, batch_sentences)
    # EN -> FR
    fr_batch = translate_batch(model_en_fr, tokenizer_en_fr, en_batch)
    return fr_batch


# -----------------------------
# GENERATE BACKTRANSLATIONS (FAST)
# -----------------------------
print(f"\nGenerating {NUM_PAIRS} backtranslated pairs...\n")

bt_sentences = []
for i in tqdm(range(0, NUM_PAIRS, BATCH_SIZE)):
    batch = original_sentences[i:i+BATCH_SIZE]
    try:
        new_batch = backtranslate_batch(batch)
        bt_sentences.extend(new_batch)
    except Exception as e:
        print("Error during batch:", e)
        bt_sentences.extend(batch)  # fallback

# -----------------------------
# SAVE OUTPUT
# -----------------------------
df_out = pd.DataFrame({
    "sentence1": original_sentences,
    "sentence2": bt_sentences,
    "similarity_score": [""] * NUM_PAIRS
})

df_out.to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print("\nDone!")
print(f"Backtranslation dataset saved as: {OUTPUT_FILE}")
print(df_out.head())


Using device: cpu

Loading dataset...

Loading translation models (FR→EN and EN→FR)...

Generating 2000 backtranslated pairs...



100%|██████████| 250/250 [56:01<00:00, 13.44s/it]  



Done!
Backtranslation dataset saved as: sts_dataset_backtranslation_vd.csv
                                           sentence1  \
0                              Tu me tel Relou Stp ?   
1  Bah ecoute jme plains pas... J'ai tellement la...   
2                     Grrrr Ã§a Frangin va le moral?   
3                Bah vo ui! t ' en dou tais e ncore?   
4  type FrÃ©rot arrÃªte d'faire le boloss devant ...   

                                           sentence2 similarity_score  
0                               Tu aimes Relou Stp ?                   
1  Je suis tellement énervée que si tu faisais un...                   
2                    Grrrr Frangin se sent-il bien ?                   
3             Tu es sûr de ne pas vouloir faire ça ?                   
4  type FrÃ©rot arrêter de faire le boloss devant...                   


In [5]:
import pandas as pd
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# SETTINGS
# -----------------------------
INPUT_FILE = "final_corpus.csv"
OUTPUT_FILE = "sts_dataset_hard_negative.csv"
NUM_PAIRS = 2000

# -----------------------------
# LOAD DATASET
# -----------------------------
print("Loading dataset...")

df = pd.read_csv(INPUT_FILE, sep=";", encoding="utf-8", on_bad_lines="skip")

if "text" not in df.columns:
    raise ValueError("The dataset must contain a column named 'text'.")

sentences = df["text"].astype(str).drop_duplicates().tolist()

# Select a subset of 10k sentences to speed up TF-IDF
subset_size = min(10000, len(sentences))
sentences_subset = random.sample(sentences, subset_size)

print(f"Using {subset_size} sentences for TF-IDF Hard Negative Mining...")

# -----------------------------
# TF-IDF VECTORIZATION
# -----------------------------
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

print("Vectorizing sentences...")
tfidf_matrix = vectorizer.fit_transform(sentences_subset)

# -----------------------------
# HARD NEGATIVE MINING
# -----------------------------
print("Computing cosine similarity...")

cosine_sim_matrix = cosine_similarity(tfidf_matrix)

pairs_sentence1 = []
pairs_sentence2 = []

for i in range(len(sentences_subset)):
    sims = cosine_sim_matrix[i]
    top_indices = sims.argsort()[::-1]  # descending order

    # Find a hard negative: lexically proche mais pas paraphrase
    for j in top_indices[1:]:
        s1 = sentences_subset[i]
        s2 = sentences_subset[j]
        # On évite les phrases identiques ou trop proches
        if s1 != s2 and s1.strip().lower() != s2.strip().lower():
            # Lexical overlap mais pas paraphrase
            if 2 <= len(set(s1.split()).intersection(set(s2.split()))) < len(set(s1.split())):
                pairs_sentence1.append(s1)
                pairs_sentence2.append(s2)
                break  # move to next sentence
    if len(pairs_sentence1) >= NUM_PAIRS:
        break

# -----------------------------
# BUILD OUTPUT CSV
# -----------------------------
output_df = pd.DataFrame({
    "sentence1": pairs_sentence1[:NUM_PAIRS],
    "sentence2": pairs_sentence2[:NUM_PAIRS],
    "similarity_score": [""] * NUM_PAIRS
})

output_df.to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print(f"\nDone! Hard negative dataset saved as {OUTPUT_FILE}")
print(output_df.head())

Loading dataset...
Using 10000 sentences for TF-IDF Hard Negative Mining...
Vectorizing sentences...
Computing cosine similarity...

Done! Hard negative dataset saved as sts_dataset_hard_negative.csv
                                           sentence1  \
0      Pitre gregaire a infrastructure de bolognaise   
1                             t pas bien ds ta tete    
2  Plan cul ne m'interesse pas.. Ce soir je peux ...   
3  Je suis sur une aire a  une heure de Nice. Jar...   
4                                 Tjs ac la CCAS?,,,   

                                           sentence2 similarity_score  
0              Pitre gregaire a  base de bolognaise                    
1                  a t pas bien bien que ds ta tete'                   
2  Ah ooi mais je peux pas t'ajooter comme ami --...                   
3              Ouai c tjs une heure de gagnee !!!,,,                   
4  Fo parler dun truc en rapport ac la nana condi...                   


In [ ]:
#Validation and cleaning of the SMS dataset

import pandas as pd

# ================================
# SETTINGS
# ================================
INPUT_FILE = "STS_dataset.csv"
OUTPUT_FILE = "STS_dataset_cleaned.csv"

MIN_CHARS = 1      # phrases trop courtes seront supprimées
ALLOW_IDENTICAL_IF_SCORE_5 = True

print("Loading dataset...")
df = pd.read_csv(INPUT_FILE, sep=";", encoding="utf-8", on_bad_lines="skip")

# Normalize column names
df.columns = df.columns.str.strip().str.lower()

# We assume columns: sentence1, sentence2, similarity_score (score can be absent)
required_cols = {"sentence1", "sentence2"}
if not required_cols.issubset(set(df.columns)):
    raise ValueError(f"Dataset must contain the columns {required_cols}")

has_score = "similarity_score" in df.columns


# ================================
# 1. CLEANING BASIC FORMAT
# ================================
df["sentence1"] = df["sentence1"].astype(str).str.strip()
df["sentence2"] = df["sentence2"].astype(str).str.strip()

initial_len = len(df)


# ================================
# 2. REMOVE EMPTY OR VERY SHORT SENTENCES
# ================================
df = df[
    (df["sentence1"].str.len() >= MIN_CHARS) &
    (df["sentence2"].str.len() >= MIN_CHARS)
 ]

print(f"- Removed short / empty sentences: {initial_len - len(df)}")


# ================================
# 3. REMOVE IDENTICAL SENTENCES
#    Allowed only if score == 5
# ================================
if has_score and ALLOW_IDENTICAL_IF_SCORE_5:
    before = len(df)

    df = df[
        ~(
            (df["sentence1"] == df["sentence2"]) &
            (df["similarity_score"] != 5)
        )
    ]

    print(f"- Removed identical pairs with score != 5: {before - len(df)}")

else:
    before = len(df)
    df = df[df["sentence1"] != df["sentence2"]]
    print(f"- Removed all identical pairs: {before - len(df)}")


# ================================
# 4. REMOVE PERFECT DUPLICATES
# ================================
before = len(df)
df = df.drop_duplicates(subset=["sentence1", "sentence2"])
print(f"- Removed exact duplicates: {before - len(df)}")


# ================================
# 5. REMOVE REVERSED DUPLICATES (A,B) vs (B,A)
# ================================
before = len(df)

# create canonical representation
df["canon_pair"] = df.apply(
    lambda x: tuple(sorted([x["sentence1"], x["sentence2"]])),
    axis=1
)

df = df.drop_duplicates(subset=["canon_pair"])
df = df.drop(columns=["canon_pair"])

print(f"- Removed reversed duplicates (A,B)/(B,A): {before - len(df)}")


# ================================
# 6. REMOVE ABSURD PAIRS
# (heuristics can be tuned)
# ================================
before = len(df)

# remove pairs where one sentence is extremely longer than the other (ratio > 5:1)
df = df[
    (df["sentence1"].str.len() / df["sentence2"].str.len()).between(1/5, 5)
]

print(f"- Removed absurd length mismatch pairs: {before - len(df)}")


# ================================
# 7. OPTIONAL: remove sentences that contain only numbers or symbols
# ================================
before = len(df)

df = df[
    df["sentence1"].str.contains(r"[A-Za-zÀ-ÿ]", regex=True) &
    df["sentence2"].str.contains(r"[A-Za-zÀ-ÿ]", regex=True)
]

print(f"- Removed non-textual pairs: {before - len(df)}")


# ================================
# SAVE RESULT
# ================================
df.to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print("\n============================================")
print("CLEANING COMPLETE")
print(f"Original dataset size : {initial_len}")
print(f"Cleaned dataset size  : {len(df)}")
print(f"Saved to              : {OUTPUT_FILE}")
print("============================================")

In [13]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------
# SETTINGS
# -----------------------------
MODEL_PATH = './flaubert_sts_final'  # Path to your fine-tuned STS model
INPUT_FILE = 'sts_augmented_without_score.csv'  # File containing sentence1 / sentence2
OUTPUT_FILE = 'sts_dataset_annotated.csv'   # Output file with annotated scores
MAX_LENGTH = 128

# -----------------------------
# LOAD MODEL + TOKENIZER
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# -----------------------------
# LOAD DATASET
# -----------------------------
print("Loading dataset...")
df = pd.read_csv(INPUT_FILE, sep=";", on_bad_lines='skip', encoding="utf-8")

# Ensure the necessary columns exist and are cleaned
df.columns = df.columns.str.lower().str.strip()

if not {'sentence1', 'sentence2'}.issubset(df.columns):
    raise ValueError("Dataset must contain 'sentence1' and 'sentence2' columns")

# Remove empty or null sentence pairs
df = df.dropna(subset=['sentence1', 'sentence2'])

# Convert sentences to strings
df['sentence1'] = df['sentence1'].astype(str)
df['sentence2'] = df['sentence2'].astype(str)

# -----------------------------
# FUNCTION TO PREDICT STS SCORE
# -----------------------------
def predict_sts_score(s1, s2):
    inputs = tokenizer(
        s1, s2,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        score = outputs.logits.squeeze().item()

    # Clamp the score between 0 and 5
    return max(0, min(5, score))

# -----------------------------
# FUNCTION TO REMOVE DUPLICATE AND INVERSE PAIRS
# -----------------------------
def remove_duplicate_and_inverse_pairs(df):
    seen_pairs = set()
    clean_pairs = []

    for index, row in df.iterrows():
        pair1 = (row['sentence1'], row['sentence2'])
        pair2 = (row['sentence2'], row['sentence1'])
        
        # Check if pair is duplicate or inverse pair
        if pair1 not in seen_pairs and pair2 not in seen_pairs:
            seen_pairs.add(pair1)
            clean_pairs.append(row)
    
    clean_df = pd.DataFrame(clean_pairs)
    clean_df.reset_index(drop=True, inplace=True)  # Reset index after cleaning
    return clean_df

# -----------------------------
# PROCESSING DATA IN BATCHES FOR EFFICIENCY
# -----------------------------
def batch_predict_sts_scores(df, batch_size=16):
    predictions = []
    
    # Process data in batches
    for i in tqdm(range(0, len(df), batch_size)):
        batch = df.iloc[i:i + batch_size]
        
        # Tokenize the batch
        inputs = tokenizer(
            batch["sentence1"].tolist(),
            batch["sentence2"].tolist(),
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            scores = outputs.logits.squeeze().cpu().numpy()
            predictions.extend(scores)

    return predictions

# -----------------------------
# PREDICT ALL SCORES IN BATCHES
# -----------------------------
print("Generating STS scores in batches...")

# Clean dataset by removing duplicate and inverse pairs
df_clean = remove_duplicate_and_inverse_pairs(df)

# Predict scores for all pairs
predicted_scores = batch_predict_sts_scores(df_clean)

# Save the results back to the dataframe
df_clean["similarity_score"] = [max(0, min(5, score)) for score in predicted_scores]

# -----------------------------
# EVALUATION AND PERFORMANCE METRICS
# -----------------------------
# Calculate MSE, MAE, RMSE, and R2
true_scores = df_clean['similarity_score'].tolist()
predicted_scores_array = np.array(predicted_scores)
true_scores_array = np.array(true_scores)

mse = mean_squared_error(true_scores_array, predicted_scores_array)
mae = mean_absolute_error(true_scores_array, predicted_scores_array)
rmse = np.sqrt(mse)
r2 = r2_score(true_scores_array, predicted_scores_array)

# Calculate Pearson and Spearman Correlations
spearman_corr, _ = spearmanr(true_scores_array, predicted_scores_array)
pearson_corr, _ = pearsonr(true_scores_array, predicted_scores_array)

print(f"\nMSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
print(f"Spearman Correlation: {spearman_corr:.4f}, Pearson Correlation: {pearson_corr:.4f}")

# -----------------------------
# SAVE RESULTS WITH PROPER FORMATTING
# -----------------------------
# Keep only the three columns and use semicolon as separator
df_clean[["sentence1", "sentence2", "similarity_score"]].to_csv(OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

print(f"\nResults saved to {OUTPUT_FILE}")

Loading model and tokenizer...
Loading dataset...
Generating STS scores in batches...
Loading dataset...
Generating STS scores in batches...


  0%|          | 2/558 [00:00<01:28,  6.27it/s]Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
  1%|▏         | 8/558 [00:00<00:43, 12.68it/s]Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be


MSE: 0.0001, MAE: 0.0002, RMSE: 0.0073, R²: 0.9998
Spearman Correlation: 1.0000, Pearson Correlation: 0.9999

Results saved to sts_dataset_annotated.csv


In [4]:
import transformers 
print (transformers.__version__)

4.57.1


In [5]:
#Phase 1 :  fine-tuning of  Flaubert-base with litlle 500 pairs dataset (sts_v0). 
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

MODEL_NAME = "flaubert/flaubert_base_cased"
OUTPUT_DIR = "./flaubert_sts_final"
MAX_LENGTH = 128

# Load and prepare the dataset
df = pd.read_csv("STS_dataset.csv", sep=";", on_bad_lines='skip', encoding="utf-8")
df = df[['sentence1', 'sentence2', 'similarity_score']].dropna()
df = df.rename(columns={'similarity_score': 'labels'})
df['labels'] = df['labels'].astype(float)

train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    result = tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    result["labels"] = examples["labels"]
    return result

train_tokenized = train_dataset.map(tokenize_fn, batched=True)
eval_tokenized = eval_dataset.map(tokenize_fn, batched=True)

# Remove unused columns
for col in ["__index_level_0__", "sentence1", "sentence2"]:
    if col in train_tokenized.column_names:
        train_tokenized = train_tokenized.remove_columns([col])
    if col in eval_tokenized.column_names:
        eval_tokenized = eval_tokenized.remove_columns([col])

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    logging_dir="./logs_sts",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Training completed. Model saved at:", OUTPUT_DIR)

Map:   0%|          | 0/402 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Some weights of FlaubertForSequenceClassification were not initialized from the model checkpoint at flaubert/flaubert_base_cased and are newly initialized: ['sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Bryan\AppData\Local\Temp\ipykernel_22996\3557571877.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,3.245800,3.818354
2,3.280400,2.607764
3,2.634400,2.510246
4,2.424800,2.118852
5,2.062400,2.037786


C:\Users\Bryan\AppData\Roaming\Python\Python313\site-packages\transformers\trainer.py:4380: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")
C:\Users\Bryan\AppData\Roaming\Python\Python313\site-packages\transformers\trainer.py:4380: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")
C:\Users\Bryan\AppData\Roaming\Python\Python313\site-packages\transformers\trainer.py:4380: UserWarning: mtime may not be reliable on this filesystem, falling back to numerical ordering
  warnings.warn("mtime may not be reliable on this filesystem, falling back to numerical ordering")


✅ Training completed. Model saved at: ./flaubert_sts_final
